# URL feature engineering and Decision Tree

Madushan S.C · URLSentra · Proposal §§4.6–4.10 and 4.12

Place the PhiUSIIL CSV in `data/raw/` and run from the repository root or `notebooks/`. The default run develops the model on a 20,000-row sample. For the final experiment, confirm the group’s feature and split decisions, set `SAMPLE = None`, and enable `RUN_FINAL_TEST`.

Phishing is class 1 after the shared target conversion. `URLSimilarityIndex` is excluded from both arms because of the group's leakage finding.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src import config, data, evaluate, features
from src.models import tree

SAMPLE = 20_000
RUN_FINAL_TEST = False

df = data.load_clean_data()
if SAMPLE is not None and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)

original = [col for col in data.get_original_features(df)
            if col not in features.EXCLUDED_FEATURES]
df = features.add_features(df)
candidates = original + features.ENGINEERED_FEATURES
X_train, X_test, y_train, y_test = data.split(df, candidates)
print(f"Training rows: {len(X_train):,} | Holdout rows: {len(X_test):,}")

## URL signals and feature selection

| Signal | What it measures |
|---|---|
| Path depth | Number of non-empty path segments |
| Longest path segment | Largest segment length, which can indicate large identifiers or slugs |
| Query field count | Number of non-empty ampersand-separated query fields |
| Encoded byte count | Number of valid percent-encoded bytes in the path and query |

These counts describe where and how the URL is structured. They use the URL string alone. The selector removes constant columns and keeps one of each pair with absolute Pearson correlation ≥ 0.98. It prefers existing columns by order. Correlation filters linear redundancy and can miss nonlinear relationships.

In [ ]:
selected = features.select_features(df.loc[X_train.index], candidates)
feature_sets = {"original": original, "engineered": selected}
display(pd.DataFrame({
    "feature": candidates,
    "engineered_url_feature": [col in features.ENGINEERED_FEATURES for col in candidates],
    "kept_in_engineered_set": [col in selected for col in candidates],
}))
display(X_train[features.ENGINEERED_FEATURES].describe().round(2))

## Training-only tuning

Choose tree settings by phishing F1 using stratified cross-validation on the original training features. Reuse these settings in both arms so that the holdout comparison changes only the feature set. The engineered arm measures the combined effect of feature creation and selection. The grid results below are development metrics. `main.py` uses the fixed starting configuration in `tree.build()`; this notebook reports the tuned comparison.

In [ ]:
search = tree.tune(X_train[original], y_train)
models = {"original": search.best_estimator_}
models["engineered"] = tree.build().set_params(**search.best_params_).fit(
    X_train[selected], y_train
)

print("Chosen settings:", search.best_params_)
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[["rank_test_score", "params", "mean_test_score", "std_test_score"]]
        .sort_values("rank_test_score").head().round(4))

## Holdout comparison

Run this after freezing the feature and split decisions. Both arms use the same held-out rows. Positive metric deltas indicate an increase in the engineered arm. Check false positives and false negatives as well as F1. Near-perfect scores warrant a check of collection bias, remaining leakage and the shared split before reporting.

In [ ]:
if RUN_FINAL_TEST:
    results = {}
    for name, cols in feature_sets.items():
        model = models[name]
        results[(name, "Decision Tree")] = evaluate.evaluate(model, X_test[cols], y_test)
        evaluate.plot_confusion_matrix(model, X_test[cols], y_test, f"Decision Tree: {name}")
    scores = evaluate.compare(results).set_index("feature_set")
    display(scores.round(4))
    metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    delta = scores.loc["engineered", metrics] - scores.loc["original", metrics]
    display(delta.astype(float).round(4).rename("Engineered minus original").to_frame())
    plt.show()
else:
    print("Holdout evaluation is disabled until the experiment settings are frozen.")

## Experiment log

| Experiment | Purpose | Evidence |
|---|---|---|
| Training CV on original features | Choose tree depth, minimum leaf size and class weighting | Generated ranking and mean phishing F1 above |
| Original versus engineered + selected | Measure changes on identical held-out URLs | Generated score deltas and confusion matrices after enabling the holdout |